# Практическая работа 3. Простые RAG-системы

В ноутбуке рассматриваются базовые этапы построения RAG-пайплайна: парсинг документов, разбиение текста на чанки, работа с векторными базами данных и сравнение LLM через API.

В рамках работы:
- реализуется собственный парсинг PDF, DOCX и TXT;
- сравниваются Qdrant и ChromaDB;
- сравниваются GigaChat и две модели OpenRouter;
- проводятся замеры скорости, памяти, стабильности и качества ответов.

## Задание 1. Собственный парсинг документов

Реализуется простой pipeline для загрузки PDF, DOCX и TXT без использования `unstructured`.

После извлечения текст очищается от лишних пробелов, лишних переносов строк и типичных артефактов PDF, а затем разбивается на чанки фиксированного размера с overlap.

In [1]:
!pip install -q pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.1 MB/s eta 0:00:00


In [2]:
import os
import re
import time

from pypdf import PdfReader
from docx import Document

import pandas as pd

In [10]:
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

Загрузка PDF, DOCX и TXT:

In [3]:
def load_pdf(path):
    reader = PdfReader(path)

    pages = []

    for page in reader.pages:
        text = page.extract_text() or ""
        pages.append(text)

    return "\n\n".join(pages)


def load_docx(path):
    doc = Document(path)

    paragraphs = [
        paragraph.text
        for paragraph in doc.paragraphs
        if paragraph.text.strip()
    ]

    return "\n".join(paragraphs)


def load_txt(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

Выполним простую очистку текста, удалив:

- лишние пробелы;
- переносы;
- битые символы;
- разрывы слов вида exam-\nple.

In [4]:
def clean_text(text):
    # Убираем нулевые и replacement-символы
    text = text.replace("\x00", "")
    text = text.replace("\ufffd", "")

    # Склеиваем слова, разорванные переносом:
    # "exam-\nple" -> "example"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Нормализуем переводы строк
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Убираем лишние пробелы и табы
    text = re.sub(r"[ \t]+", " ", text)

    # Не более двух последовательных переносов строк
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Убираем пробелы по краям строк
    text = "\n".join(line.strip() for line in text.splitlines())

    return text.strip()

Единый load_document(path):

In [5]:
def load_document(path):
    extension = os.path.splitext(path)[1].lower()

    if extension == ".pdf":
        text = load_pdf(path)

    elif extension == ".docx":
        text = load_docx(path)

    elif extension == ".txt":
        text = load_txt(path)

    else:
        raise ValueError(f"Неподдерживаемый формат: {extension}")

    return clean_text(text)

Разбиение на чанки. Возьмём разумные значения из примера в задании:

- chunk_size = 800 символов
- overlap = 150 символов

In [6]:
def split_into_chunks(
    text,
    chunk_size=800,
    overlap=150,
):
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size")

    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

Проверим pipeline на одном PDF:

In [7]:
path = "/content/data/0007-pdf.pdf"

start = time.perf_counter()

text = load_document(path)

parse_time = time.perf_counter() - start

chunks = split_into_chunks(text)

print("Файл:", os.path.basename(path))
print("Время парсинга:", round(parse_time, 3), "сек.")
print("Объём текста:", len(text), "символов")
print("Количество чанков:", len(chunks))

print("\nПервые 1500 символов:\n")
print(text[:1500])

Файл: 0007-pdf.pdf
Время парсинга: 0.148 сек.
Объём текста: 9911 символов
Количество чанков: 16

Первые 1500 символов:

Information Security Checks (March) 2016/17
City of York Council
Internal Audit Report





Service Area: Corporate and Cross-Cutting
Responsible Officer: Assistant Director, Legal and Governance
Service Manager: Information Governance and Feedback Team Manager
Version: Final
Date Issued: 1 June 2017
Reference: 10260/022

Overall Audit Opinion Reasonable Assurance
Actions
P1 P2 P3
0


3 0

Summary and Overall Conclusions
Introduction and objectives
1.1 In accordance with the agreed audit plan, information security checks were undertaken during 2016/17. The purpose of these checks is to
assess the extent to which confidential, personal or sensitive data is stored securely and to ensure that data security is being given
sufficient priority within council departments. This was the second of these checks in this audit year (the first being in September 2016).

1.2 The pre

In [8]:
print(chunks[0])

Information Security Checks (March) 2016/17
City of York Council
Internal Audit Report





Service Area: Corporate and Cross-Cutting
Responsible Officer: Assistant Director, Legal and Governance
Service Manager: Information Governance and Feedback Team Manager
Version: Final
Date Issued: 1 June 2017
Reference: 10260/022

Overall Audit Opinion Reasonable Assurance
Actions
P1 P2 P3
0


3 0

Summary and Overall Conclusions
Introduction and objectives
1.1 In accordance with the agreed audit plan, information security checks were undertaken during 2016/17. The purpose of these checks is to
assess the extent to which confidential, personal or sensitive data is stored securely and to ensure that data security is being given
sufficient priority within council departments. This was the second of t


По результату уже видно несколько важных вещей:

- текст извлекается успешно и быстро: 0.148 с;
- получено 9911 символов и 16 чанков;
- обычный текст читается нормально;
- структура таблицы в начале документа уже частично потеряна: блок P1 P2 P3 / 0 / 3 0 распался на отдельные строки;
- встречается типичный PDF-артефакт вроде informatio n security, то есть разрыв слова без дефиса текущая очистка пока не исправляет.

Прогоним все файлы и соберём метрики:

In [11]:
FILES = [
    "0007-pdf.pdf",
    "0008-pdf.pdf",
    "0009-pdf.pdf",
    "0050-docx.docx",
    "0082-docx.docx",
    "0003-txt.txt",
]

results = []

for filename in FILES:
    path = os.path.join(DATA_DIR, filename)

    start = time.perf_counter()

    text = load_document(path)

    parse_time = time.perf_counter() - start

    chunks = split_into_chunks(text)

    results.append({
        "file": filename,
        "type": os.path.splitext(filename)[1].lower(),
        "parse_time_sec": parse_time,
        "text_chars": len(text),
        "chunks": len(chunks),
    })

metrics_df = pd.DataFrame(results)

metrics_df

,file,type,parse_time_sec,text_chars,chunks
0,0007-pdf.pdf,.pdf,0.153389,9911,16
1,0008-pdf.pdf,.pdf,0.404209,116462,180
2,0009-pdf.pdf,.pdf,0.726235,25524,40
3,0050-docx.docx,.docx,0.147107,73690,114
4,0082-docx.docx,.docx,0.090766,40824,63
5,0003-txt.txt,.txt,0.002015,13623,21


Выведем небольшие фрагменты для визуальной оценки качества:

In [12]:
for filename in FILES:
    path = os.path.join(DATA_DIR, filename)

    text = load_document(path)

    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)
    print(text[:1000])


0007-pdf.pdf
Information Security Checks (March) 2016/17
City of York Council
Internal Audit Report





Service Area: Corporate and Cross-Cutting
Responsible Officer: Assistant Director, Legal and Governance
Service Manager: Information Governance and Feedback Team Manager
Version: Final
Date Issued: 1 June 2017
Reference: 10260/022

Overall Audit Opinion Reasonable Assurance
Actions
P1 P2 P3
0


3 0

Summary and Overall Conclusions
Introduction and objectives
1.1 In accordance with the agreed audit plan, information security checks were undertaken during 2016/17. The purpose of these checks is to
assess the extent to which confidential, personal or sensitive data is stored securely and to ensure that data security is being given
sufficient priority within council departments. This was the second of these checks in this audit year (the first being in September 2016).

1.2 The previous information security checks were conducted in September and gave an overall opinion of Reasonable As

- TXT извлекается наиболее чисто: структура абзацев сохраняется, артефактов почти нет.
- DOCX тоже извлекается хорошо, но мы берём только текст абзацев, поэтому исходное форматирование, таблицы и часть служебной структуры теряются.
- PDF извлекается заметно хуже:
  - у 0007-pdf.pdf распалась табличная структура;
  - у 0008-pdf.pdf текст в целом читаемый, но видны переносы строк и артефакты вроде склеенных/разорванных слов;
  - у 0009-pdf.pdf сложная вёрстка приводит к большому количеству пустых строк и визуально «рваному» тексту.
  
По скорости всё очень быстро: от 0.002 с для TXT до 0.726 с для самого сложного PDF. По объёму текста больше всего дал 0008-pdf.pdf — 116 462 символа.

### Сравнение с Unstructured

Те же документы обрабатываются через Unstructured API.

Для сравнения фиксируются:
- время парсинга;
- объём извлечённого текста;
- количество чанков;
- качество сохранения структуры документа.

In [23]:
!pip install -q unstructured-transform-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.1/94.1 kB 2.4 MB/s eta 0:00:00


In [19]:
from google.colab import userdata

UNSTRUCTURED_API_KEY = userdata.get("UNSTRUCTURED_API_KEY")

In [24]:
from unstructured_transform_client import TransformClient

client = TransformClient(
    api_key=UNSTRUCTURED_API_KEY,
    server_url="https://transform.unstructured.io",
)

In [25]:
def load_with_unstructured(path):
    with open(path, "rb") as f:
        result = client.parse.run(input=f)

    return result.markdown

Unstructured не поддерживает txt, падает с ошибкой, поэтому здесь сравнение проведу только для PDF и DOCX.

In [28]:
UNSTRUCTURED_FILES = [
    "0007-pdf.pdf",
    "0008-pdf.pdf",
    "0009-pdf.pdf",
    "0050-docx.docx",
    "0082-docx.docx",
]

In [30]:
unstructured_results = []
unstructured_texts = {}

for filename in UNSTRUCTURED_FILES:
    path = os.path.join(DATA_DIR, filename)

    start = time.perf_counter()

    text = load_with_unstructured(path)

    parse_time = time.perf_counter() - start

    chunks = split_into_chunks(text)

    unstructured_texts[filename] = text

    unstructured_results.append({
        "file": filename,
        "type": os.path.splitext(filename)[1].lower(),
        "parse_time_sec": parse_time,
        "text_chars": len(text),
        "chunks": len(chunks),
    })

In [31]:
unstructured_df = pd.DataFrame(unstructured_results)
unstructured_df

,file,type,parse_time_sec,text_chars,chunks
0,0007-pdf.pdf,.pdf,30.312340,10245,16
1,0008-pdf.pdf,.pdf,84.010422,119235,184
2,0009-pdf.pdf,.pdf,50.364800,25981,40
3,0050-docx.docx,.docx,3.104502,74264,115
4,0082-docx.docx,.docx,1.640905,41056,64


In [32]:
custom_compare_df = metrics_df[
    metrics_df["file"].isin(UNSTRUCTURED_FILES)
]

comparison_df = custom_compare_df.merge(
    unstructured_df,
    on=["file", "type"],
    suffixes=("_custom", "_unstructured"),
)

comparison_df

,file,type,parse_time_sec_custom,text_chars_custom,chunks_custom,parse_time_sec_unstructured,text_chars_unstructured,chunks_unstructured
0,0007-pdf.pdf,.pdf,0.153389,9911,16,30.312340,10245,16
1,0008-pdf.pdf,.pdf,0.404209,116462,180,84.010422,119235,184
2,0009-pdf.pdf,.pdf,0.726235,25524,40,50.364800,25981,40
3,0050-docx.docx,.docx,0.147107,73690,114,3.104502,74264,115
4,0082-docx.docx,.docx,0.090766,40824,63,1.640905,41056,64


In [33]:
for filename in UNSTRUCTURED_FILES:
    text = unstructured_texts[filename]

    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)
    print(text[:1000])


0007-pdf.pdf
Veritau
Assurance Services for
the Public Sector

# Information Security Checks (March) 2016/17
City of York Council
Internal Audit Report

Service Area: Corporate and Cross-Cutting
Responsible Officer: Assistant Director, Legal and Governance
Service Manager: Information Governance and Feedback Team Manager
Version: Final
Date Issued: 1 June 2017
Reference: 10260/022

<table><tr><th></th><th>P1</th><th>P2</th><th>P3</th></tr><tr><td>Actions</td><td>0</td><td>3</td><td>0</td></tr><tr><td>Overall Audit Opinion</td><td colspan="3">Reasonable Assurance</td></tr></table>

# Summary and Overall Conclusions

# Introduction and objectives

1.1    In accordance with the agreed audit plan, information security checks were undertaken during 2016/17. The purpose of these checks is to
assess the extent to which confidential, personal or sensitive data is stored securely and to ensure that data security is being given
sufficient priority within council departments. This was the second

### Вывод

Собственный парсер успешно обработал PDF, DOCX и TXT и оказался значительно быстрее Unstructured. Для тестовых файлов время обработки составляло примерно 0.09–0.73 с, тогда как Unstructured требовал от 1.6 до 84 с.

Для обычного линейного текста собственный парсер показывает хорошее качество. Особенно хорошо обрабатываются TXT и DOCX. При работе с PDF заметны типичные ограничения: теряется исходное форматирование, таблицы могут превращаться в последовательность строк, а сложная вёрстка приводит к лишним переносам и другим артефактам.

Unstructured лучше сохраняет структуру документа. Например, таблица в `0007-pdf.pdf` была восстановлена в виде HTML-таблицы, а заголовки, списки и отдельные элементы сложной вёрстки в PDF распознавались более явно. При этом объём извлечённого текста у двух подходов оказался близким.

Таким образом:
- собственный парсер быстрее, проще и не требует внешнего API;
- Unstructured лучше работает со сложной структурой, таблицами и форматированием;
- собственный вариант подходит для простых и предсказуемых документов, особенно если важны скорость и отсутствие внешних зависимостей;
- для универсального production-пайплайна с разнородными и сложными документами предпочтительнее использовать специализированный инструмент вроде Unstructured либо дополнительно развивать собственный парсер.

## Задание 2. Сравнение Qdrant и ChromaDB

Проводится сравнение двух векторных баз данных на одинаковых embeddings и одинаковых наборах документов.

Создаются две коллекции:
- `collection_small` — 1000 документов;
- `collection_large` — 7500 документов.

Для обеих БД используются одинаковая размерность векторов и cosine distance. Сравниваются скорость индексации и поиска, потребление RAM, размер хранилища, стабильность retrieval и поведение при последовательной и параллельной нагрузке.

In [34]:
!pip install -q qdrant-client chromadb sentence-transformers psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [35]:
import os
import time
import shutil
import psutil
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import chromadb

In [36]:
SMALL_SIZE = 1000
LARGE_SIZE = 7500

QDRANT_PATH = "/content/qdrant_storage"
CHROMA_PATH = "/content/chroma_storage"

Возьмём небольшую популярную модель:

In [37]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

VECTOR_SIZE = embedding_model.get_sentence_embedding_dimension()

print("Vector size:", VECTOR_SIZE)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector size: 384


/tmp/ipykernel_2030/3131104053.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_SIZE = embedding_model.get_sentence_embedding_dimension()


Возьмём тексты из предыдущего задания:

In [38]:
all_chunks = []

for filename in FILES:
    path = os.path.join(DATA_DIR, filename)

    text = load_document(path)
    chunks = split_into_chunks(text)

    all_chunks.extend(chunks)

print("Исходных чанков:", len(all_chunks))

Исходных чанков: 434


Потом создадим нужное количество записей простым повторением, но сделаем записи формально уникальными, добавив Document ID.

In [39]:
def make_dataset(chunks, size):
    docs = []

    for i in range(size):
        base_text = chunks[i % len(chunks)]
        docs.append(f"{base_text}\nDocument ID: {i}")

    return docs

In [40]:
docs_small = make_dataset(all_chunks, SMALL_SIZE)
docs_large = make_dataset(all_chunks, LARGE_SIZE)

print("Small:", len(docs_small))
print("Large:", len(docs_large))

Small: 1000
Large: 7500


Считаем эмбеддинги один раз:

In [42]:
start = time.perf_counter()

embeddings_small = embedding_model.encode(
    docs_small,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings_large = embedding_model.encode(
    docs_large,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embedding_time = time.perf_counter() - start

print("Embedding time:", round(embedding_time, 2), "sec")
print("Small shape:", embeddings_small.shape)
print("Large shape:", embeddings_large.shape)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Embedding time: 620.17 sec
Small shape: (1000, 384)
Large shape: (7500, 384)


### Qdrant

Создаются две локальные коллекции Qdrant с cosine distance и одинаковой размерностью векторов. Измеряются время индексации, изменение потребления RAM и размер хранилища на диске.

In [43]:
# вспомогательные функции

def get_ram_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024


def get_dir_size_mb(path):
    total = 0

    for root, dirs, files in os.walk(path):
        for file in files:
            filepath = os.path.join(root, file)
            total += os.path.getsize(filepath)

    return total / 1024 / 1024

In [44]:
# очистка старого хранилища на всякий случай

if os.path.exists(QDRANT_PATH):
    shutil.rmtree(QDRANT_PATH)

qdrant = QdrantClient(path=QDRANT_PATH)

Функция создания и заполнения коллекции:

In [45]:
def benchmark_qdrant_indexing(
    collection_name,
    documents,
    embeddings,
):
    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE,
        ),
    )

    ram_before = get_ram_mb()
    start = time.perf_counter()

    qdrant.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=i,
                vector=embeddings[i].tolist(),
                payload={"text": documents[i]},
            )
            for i in range(len(documents))
        ],
    )

    elapsed = time.perf_counter() - start
    ram_after = get_ram_mb()

    return {
        "collection": collection_name,
        "documents": len(documents),
        "index_time_sec": elapsed,
        "ram_before_mb": ram_before,
        "ram_after_mb": ram_after,
        "ram_increase_mb": ram_after - ram_before,
    }

In [46]:
# индексируем small и large

qdrant_results = []

qdrant_results.append(
    benchmark_qdrant_indexing(
        "collection_small",
        docs_small,
        embeddings_small,
    )
)

qdrant_results.append(
    benchmark_qdrant_indexing(
        "collection_large",
        docs_large,
        embeddings_large,
    )
)

qdrant_index_df = pd.DataFrame(qdrant_results)
qdrant_index_df

,collection,documents,index_time_sec,ram_before_mb,ram_after_mb,ram_increase_mb
0,collection_small,1000,11.894831,1036.957031,1048.722656,11.765625
1,collection_large,7500,80.362706,1048.722656,1134.796875,86.074219


In [47]:
qdrant_disk_mb = get_dir_size_mb(QDRANT_PATH)

print("Qdrant storage size:", round(qdrant_disk_mb, 2), "MB")

Qdrant storage size: 37.77 MB


In [48]:
for collection_name in [
    "collection_small",
    "collection_large",
]:
    info = qdrant.get_collection(collection_name)

    print(
        collection_name,
        "points:",
        info.points_count,
    )

collection_small points: 1000
collection_large points: 7500


### Поиск в Qdrant

Для обеих коллекций выполняются 15 одинаковых поисковых запросов.  
Измеряется среднее время retrieval и проверяется стабильность результатов при повторении одинаковых запросов.

Чтобы не придумывать искусственные запросы вручную, возьмём 15 разных исходных чанков, т.к. сейчас мы просто измеряем производительность, а не качество извлечения.

In [49]:
query_indices = np.linspace(
    0,
    len(all_chunks) - 1,
    15,
    dtype=int,
)

query_texts = [all_chunks[i] for i in query_indices]

query_embeddings = embedding_model.encode(query_texts, normalize_embeddings=True)

print("Queries:", len(query_texts))
print("Shape:", query_embeddings.shape)

Queries: 15
Shape: (15, 384)


In [51]:
def qdrant_search(
    collection_name,
    query_vector,
    limit=5,
):
    result = qdrant.query_points(
        collection_name=collection_name,
        query=query_vector.tolist(),
        limit=limit,
    )

    return result.points

Замеряем 15 запросов:

In [52]:
def benchmark_qdrant_search(
    collection_name,
    query_embeddings,
):
    latencies = []
    result_ids = []

    for query_vector in query_embeddings:
        start = time.perf_counter()

        results = qdrant_search(collection_name, query_vector)

        latency = time.perf_counter() - start

        latencies.append(latency)

        result_ids.append([point.id for point in results])

    return {
        "collection": collection_name,
        "queries": len(query_embeddings),
        "avg_retrieval_ms": np.mean(latencies) * 1000,
        "min_retrieval_ms": np.min(latencies) * 1000,
        "max_retrieval_ms": np.max(latencies) * 1000,
        "result_ids": result_ids,
    }

In [53]:
qdrant_search_small = benchmark_qdrant_search(
    "collection_small",
    query_embeddings,
)

qdrant_search_large = benchmark_qdrant_search(
    "collection_large",
    query_embeddings,
)

qdrant_search_df = pd.DataFrame([
    {
        k: v
        for k, v in result.items()
        if k != "result_ids"
    }
    for result in [
        qdrant_search_small,
        qdrant_search_large,
    ]
])

qdrant_search_df

,collection,queries,avg_retrieval_ms,min_retrieval_ms,max_retrieval_ms
0,collection_small,15,8.953689,4.178889,26.156542
1,collection_large,15,38.261839,24.347573,73.273649


Теперь проверка стабильности - повторим все 15 запросов 3 раза.

In [54]:
def check_qdrant_stability(
    collection_name,
    query_embeddings,
    repeats=3,
):
    all_runs = []

    for _ in range(repeats):
        run = []

        for query_vector in query_embeddings:
            results = qdrant_search(collection_name, query_vector)

            run.append([point.id for point in results])

        all_runs.append(run)

    stable_queries = 0

    for i in range(len(query_embeddings)):
        first_result = all_runs[0][i]

        if all(run[i] == first_result for run in all_runs[1:]):
            stable_queries += 1

    return stable_queries / len(query_embeddings)

In [55]:
small_stability = check_qdrant_stability(
    "collection_small",
    query_embeddings,
)

large_stability = check_qdrant_stability(
    "collection_large",
    query_embeddings,
)

print(
    "Small stability:",
    f"{small_stability * 100:.1f}%"
)

print(
    "Large stability:",
    f"{large_stability * 100:.1f}%"
)

Small stability: 100.0%
Large stability: 100.0%


### Стресс-тест Qdrant

Проводится серия из 20 последовательных и 5 параллельных запросов к большой коллекции. Сравнивается средняя задержка и общее время выполнения.

In [56]:
from concurrent.futures import ThreadPoolExecutor

20 последовательных запросов:

In [59]:
def qdrant_sequential_test(
    collection_name,
    query_vectors,
    n_queries=20,
):
    latencies = []

    start_total = time.perf_counter()

    for i in range(n_queries):
        query_vector = query_vectors[i % len(query_vectors)]

        start = time.perf_counter()

        qdrant_search(collection_name, query_vector)

        latencies.append(time.perf_counter() - start)

    total_time = time.perf_counter() - start_total

    return {
        "mode": "sequential",
        "queries": n_queries,
        "avg_latency_ms": np.mean(latencies) * 1000,
        "total_time_sec": total_time,
    }

5 параллельных запросов:

In [60]:
def qdrant_parallel_test(
    collection_name,
    query_vectors,
    n_queries=5,
):
    def run_query(query_vector):
        start = time.perf_counter()

        qdrant_search(collection_name, query_vector)

        return time.perf_counter() - start

    vectors = [query_vectors[i % len(query_vectors)] for i in range(n_queries)]

    start_total = time.perf_counter()

    with ThreadPoolExecutor(max_workers=n_queries) as executor:

        latencies = list(executor.map(run_query, vectors))

    total_time = time.perf_counter() - start_total

    return {
        "mode": "parallel",
        "queries": n_queries,
        "avg_latency_ms": np.mean(latencies) * 1000,
        "total_time_sec": total_time,
    }

In [61]:
qdrant_stress_results = [
    qdrant_sequential_test(
        "collection_large",
        query_embeddings,
        n_queries=20,
    ),
    qdrant_parallel_test(
        "collection_large",
        query_embeddings,
        n_queries=5,
    ),
]

qdrant_stress_df = pd.DataFrame(qdrant_stress_results)
qdrant_stress_df

,mode,queries,avg_latency_ms,total_time_sec
0,sequential,20,7.345621,0.146996
1,parallel,5,100.029637,0.125243


### ChromaDB

Создаются две локальные коллекции ChromaDB с теми же embeddings и cosine distance.  
Измеряются время индексации, потребление RAM, размер хранилища, скорость retrieval, стабильность результатов и поведение при последовательной и параллельной нагрузке.

In [94]:
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

chroma = chromadb.PersistentClient(
    path=CHROMA_PATH
)

In [95]:
collection_small_chroma = chroma.create_collection(
    name="collection_small",
    metadata={"hnsw:space": "cosine"},
)

collection_large_chroma = chroma.create_collection(
    name="collection_large",
    metadata={"hnsw:space": "cosine"},
)

Функция индексации:

In [96]:
def benchmark_chroma_indexing(
    collection,
    collection_name,
    documents,
    embeddings,
    batch_size=1000,
):
    ram_before = get_ram_mb()

    start = time.perf_counter()

    for start_idx in range(0, len(documents), batch_size):
        end_idx = min(start_idx + batch_size, len(documents))

        collection.add(
            ids=[
                str(i)
                for i in range(start_idx, end_idx)
            ],
            documents=documents[start_idx:end_idx],
            embeddings=embeddings[start_idx:end_idx].tolist(),
        )

    elapsed = time.perf_counter() - start

    ram_after = get_ram_mb()

    return {
        "collection": collection_name,
        "documents": len(documents),
        "index_time_sec": elapsed,
        "ram_before_mb": ram_before,
        "ram_after_mb": ram_after,
        "ram_increase_mb": ram_after - ram_before,
    }

In [97]:
chroma_results = []

chroma_results.append(
    benchmark_chroma_indexing(
        collection_small_chroma,
        "collection_small",
        docs_small,
        embeddings_small,
    )
)

chroma_results.append(
    benchmark_chroma_indexing(
        collection_large_chroma,
        "collection_large",
        docs_large,
        embeddings_large,
    )
)

chroma_index_df = pd.DataFrame(chroma_results)
chroma_index_df

,collection,documents,index_time_sec,ram_before_mb,ram_after_mb,ram_increase_mb
0,collection_small,1000,0.783318,1358.296875,1363.390625,5.093750
1,collection_large,7500,7.461037,1363.390625,1365.285156,1.894531


In [98]:
chroma_disk_mb = get_dir_size_mb(CHROMA_PATH)

print(
    "Chroma storage size:",
    round(chroma_disk_mb, 2),
    "MB"
)

Chroma storage size: 63.64 MB


In [99]:
print("collection_small points:", collection_small_chroma.count())

print("collection_large points:", collection_large_chroma.count())

collection_small points: 1000
collection_large points: 7500


Уже видно заметную разницу с Qdrant:

- collection_small: 0.78 с против 11.89 с у Qdrant;
- collection_large: 7.46 с против 80.36 с;
- Chroma заняла больше места на диске: 63.64 MB против 37.77 MB у Qdrant.

### Поиск в ChromaDB

Для обеих коллекций выполняются те же 15 поисковых запросов.  
Измеряется среднее время retrieval и проверяется стабильность результатов при повторных запросах.

In [100]:
def chroma_search(
    collection,
    query_vector,
    limit=5,
):
    result = collection.query(query_embeddings=[query_vector.tolist()], n_results=limit)

    return result["ids"][0]

In [101]:
def benchmark_chroma_search(
    collection,
    collection_name,
    query_embeddings,
):
    latencies = []
    result_ids = []

    for query_vector in query_embeddings:
        start = time.perf_counter()

        ids = chroma_search(collection, query_vector)

        latency = time.perf_counter() - start

        latencies.append(latency)
        result_ids.append(ids)

    return {
        "collection": collection_name,
        "queries": len(query_embeddings),
        "avg_retrieval_ms": np.mean(latencies) * 1000,
        "min_retrieval_ms": np.min(latencies) * 1000,
        "max_retrieval_ms": np.max(latencies) * 1000,
        "result_ids": result_ids,
    }

In [102]:
chroma_search_small = benchmark_chroma_search(
    collection_small_chroma,
    "collection_small",
    query_embeddings,
)

chroma_search_large = benchmark_chroma_search(
    collection_large_chroma,
    "collection_large",
    query_embeddings,
)

chroma_search_df = pd.DataFrame([
    {
        k: v
        for k, v in result.items()
        if k != "result_ids"
    }
    for result in [
        chroma_search_small,
        chroma_search_large,
    ]
])

chroma_search_df

,collection,queries,avg_retrieval_ms,min_retrieval_ms,max_retrieval_ms
0,collection_small,15,1.737194,1.158710,4.349009
1,collection_large,15,1.390324,1.244729,1.905949


Проверка стабильности:

In [103]:
def check_chroma_stability(
    collection,
    query_embeddings,
    repeats=3,
):
    all_runs = []

    for _ in range(repeats):
        run = []

        for query_vector in query_embeddings:
            ids = chroma_search(collection, query_vector)

            run.append(ids)

        all_runs.append(run)

    stable_queries = 0

    for i in range(len(query_embeddings)):
        first_result = all_runs[0][i]

        if all(run[i] == first_result for run in all_runs[1:]):
            stable_queries += 1

    return stable_queries / len(query_embeddings)

In [104]:
small_stability_chroma = check_chroma_stability(
    collection_small_chroma,
    query_embeddings,
)

large_stability_chroma = check_chroma_stability(
    collection_large_chroma,
    query_embeddings,
)

print(
    "Small stability:",
    f"{small_stability_chroma * 100:.1f}%"
)

print(
    "Large stability:",
    f"{large_stability_chroma * 100:.1f}%"
)

Small stability: 100.0%
Large stability: 100.0%


### Стресс-тест ChromaDB

Проводится серия из 20 последовательных и 5 параллельных запросов к большой коллекции. Сравниваются средняя задержка и общее время выполнения.

In [105]:
def chroma_sequential_test(
    collection,
    query_vectors,
    n_queries=20,
):
    latencies = []

    start_total = time.perf_counter()

    for i in range(n_queries):
        query_vector = query_vectors[i % len(query_vectors)]

        start = time.perf_counter()

        chroma_search(collection, query_vector)

        latencies.append(time.perf_counter() - start)

    total_time = time.perf_counter() - start_total

    return {
        "mode": "sequential",
        "queries": n_queries,
        "avg_latency_ms": np.mean(latencies) * 1000,
        "total_time_sec": total_time,
    }

In [106]:
def chroma_parallel_test(
    collection,
    query_vectors,
    n_queries=5,
):
    def run_query(query_vector):
        start = time.perf_counter()

        chroma_search(collection, query_vector)

        return time.perf_counter() - start

    vectors = [query_vectors[i % len(query_vectors)] for i in range(n_queries)]

    start_total = time.perf_counter()

    with ThreadPoolExecutor(max_workers=n_queries) as executor:
        latencies = list(executor.map(run_query, vectors))

    total_time = time.perf_counter() - start_total

    return {
        "mode": "parallel",
        "queries": n_queries,
        "avg_latency_ms": np.mean(latencies) * 1000,
        "total_time_sec": total_time,
    }

In [107]:
chroma_stress_results = [
    chroma_sequential_test(
        collection_large_chroma,
        query_embeddings,
        n_queries=20,
    ),
    chroma_parallel_test(
        collection_large_chroma,
        query_embeddings,
        n_queries=5,
    ),
]

chroma_stress_df = pd.DataFrame(chroma_stress_results)

chroma_stress_df

,mode,queries,avg_latency_ms,total_time_sec
0,sequential,20,5.418016,0.108512
1,parallel,5,12.613202,0.019171


### Итоговое сравнение Qdrant и ChromaDB

| Метрика | Qdrant | ChromaDB |
|---|---:|---:|
| Small collection | 1000 | 1000 |
| Large collection | 7500 | 7500 |
| Размер вектора | 384 | 384 |
| Метрика | Cosine | Cosine |
| Индексация small | 11.89 с | 0.78 с |
| Индексация large | 80.36 с | 7.46 с |
| RAM прирост small | 11.77 MB | 5.09 MB |
| RAM прирост large | 86.07 MB | 1.89 MB |
| Размер хранилища | 37.77 MB | 63.64 MB |
| Retrieval small | 8.95 мс | 1.74 мс |
| Retrieval large | 38.26 мс | 1.39 мс |
| Стабильность small | 100% | 100% |
| Стабильность large | 100% | 100% |
| 20 последовательных запросов | 7.35 мс | 5.42 мс |
| 5 параллельных запросов | 100.03 мс | 12.61 мс |

### Вывод

В проведённом эксперименте ChromaDB показала более высокую скорость индексации и поиска. Для коллекции из 7500 документов индексация заняла 7.46 с против 80.36 с у Qdrant, а среднее время retrieval составило 1.39 мс против 38.26 мс. Обе базы показали 100% стабильность результатов.

При стресс-тесте ChromaDB также показала меньшую задержку как для последовательных, так и для параллельных запросов. При этом Qdrant использовал меньше дискового пространства: 37.77 MB против 63.64 MB.

Для небольших локальных проектов ChromaDB выглядит более удобным вариантом: API простой, локальный запуск не требует отдельного сервера, а в данном эксперименте поиск и индексация оказались быстрее.

Qdrant предоставляет более развитые возможности для production-сценариев: отдельный серверный режим, масштабирование и более гибкую работу с payload и фильтрами. Local mode, использованный в эксперименте, не отражает полностью производительность серверного Qdrant.

Поэтому результаты этого теста показывают преимущество ChromaDB именно для небольшого локального сценария. Для более крупных production-систем Qdrant потенциально лучше масштабируется, однако это требует отдельного тестирования в серверной конфигурации.

## Задание 3. Сравнение GigaChat и OpenRouter

Сравниваются GigaChat и две модели OpenRouter на одинаковых задачах.

Для всех моделей используется единый интерфейс вызова. Измеряются время ответа, длина ответа и количество токенов. Дополнительно проверяются длинный контекст, стабильность API и поведение при серии последовательных запросов.

In [108]:
!pip install -q openai gigachat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 3.7 MB/s eta 0:00:00


In [109]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
GIGACHAT_CREDENTIALS = userdata.get("GIGACHAT_CREDENTIALS")

In [112]:
from openai import OpenAI

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

In [131]:
OPENROUTER_SMALL = "cohere/north-mini-code:free"
OPENROUTER_LARGE = "nvidia/nemotron-3-super-120b-a12b:free"

In [114]:
from gigachat import GigaChat

gigachat_client = GigaChat(
    credentials=GIGACHAT_CREDENTIALS,
    scope="GIGACHAT_API_PERS",
    base_url="https://api.giga.chat/v1",
    verify_ssl_certs=False,
)

In [115]:
GIGACHAT_MODEL = "GigaChat-2-Max"

Напишем функции для вызова моделей с ответами в едином формате:

In [132]:
def call_openrouter(model, prompt, max_tokens=500):
    start = time.perf_counter()

    response = openrouter_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=max_tokens,
    )

    elapsed = time.perf_counter() - start

    text = response.choices[0].message.content

    if not text:
        text = "[Финальный ответ отсутствует]"

    return {
        "text": text,
        "time_sec": elapsed,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }

In [117]:
def call_gigachat(prompt, max_tokens=500):
    start = time.perf_counter()

    response = gigachat_client.chat(
        {
            "model": GIGACHAT_MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "temperature": 0,
            "max_tokens": max_tokens,
        }
    )

    elapsed = time.perf_counter() - start

    text = response.choices[0].message.content

    usage = response.usage

    return {
        "text": text,
        "time_sec": elapsed,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
    }

И реализуем абстракцию переключения моделей (единый интерфейс вызова):

In [133]:
def call_model(model_name, prompt, max_tokens=500):

    if model_name == "gigachat":
        return call_gigachat(prompt, max_tokens=max_tokens)

    elif model_name == "openrouter_small":
        return call_openrouter(OPENROUTER_SMALL, prompt, max_tokens=max_tokens)

    elif model_name == "openrouter_large":
        return call_openrouter(OPENROUTER_LARGE, prompt, max_tokens=max_tokens)

    else:
        raise ValueError(f"Unknown model: {model_name}")

Проверка подключения на примере одного запроса:

In [135]:
test_prompt = "Ответь одним предложением: что такое RAG?"

for model_name in [
    "gigachat",
    "openrouter_small",
    "openrouter_large",
]:
    print("\n" + "=" * 80)
    print(model_name)

    result = call_model(
        model_name,
        test_prompt,
        max_tokens=500,
    )

    print("Time:", round(result["time_sec"], 2), "sec")
    print("Prompt tokens:", result["prompt_tokens"])
    print("Completion tokens:", result["completion_tokens"])
    print("Total tokens:", result["total_tokens"])
    print("Answer:", result["text"])


gigachat
Time: 1.45 sec
Prompt tokens: 26
Completion tokens: 40
Total tokens: 66
Answer: RAG (Retrieval-Augmented Generation) — это технология, объединяющая методы извлечения информации из внешних источников с генерацией текста моделями машинного обучения для повышения точности и надежности ответов.

openrouter_small
Time: 0.85 sec
Prompt tokens: 11
Completion tokens: 137
Total tokens: 148
Answer: RAG (Retrieval-Augmented Generation) — это гибридный подход к ИИ, который извлекает релевантные внешние знания и использует их для улучшения точности и связности генерируемых ответов.

openrouter_large
Time: 3.59 sec
Prompt tokens: 28
Completion tokens: 140
Total tokens: 168
Answer: RAG (Retrieval‑Augmented Generation) — это подход, при котором генеративная модель дополняет свои ответы информацией, извлечённой из внешних источников, чтобы повысить их точность и релевантность.


### Основные тесты моделей

Три модели тестируются на одинаковых задачах:

- логическая задача;
- генерация кода;
- аналитический ответ;
- RAG-вопрос по документу.

Для каждого ответа измеряются время генерации, длина ответа и количество токенов.

In [142]:
logic_prompt = """
У Маши, Олега и Иры разные профессии: программист, учитель и врач.

Известно:
- Маша не врач.
- Олег не программист.
- Ира не учитель.
- Учитель — не Маша.

Определи профессию каждого человека.
Кратко объясни ход рассуждений.
"""

code_prompt = """
Напиши функцию Python second_largest_unique(numbers),
которая возвращает второе по величине уникальное число в списке.

Требования:
- дубликаты не должны влиять на результат;
- если уникальных чисел меньше двух, вернуть None;
- приведи код и короткое объяснение.
"""

analytic_prompt = """
Объясни, зачем в машинном обучении данные обычно разделяют
на train, validation и test.

Ответ должен содержать:
1. назначение каждой выборки;
2. почему нельзя постоянно подбирать параметры по test;
3. короткий практический пример.

Ответ дай на русском языке.
"""

Для RAG возьмём документ из первого задания. Сформируем контекст из начала документа:

In [136]:
rag_path = os.path.join(DATA_DIR, "0008-pdf.pdf")
rag_text = load_document(rag_path)
rag_context = rag_text[:6000]

In [139]:
rag_prompt = f"""
Ответь на вопрос, используя только приведённый контекст.
Если ответа в контексте нет, прямо скажи об этом.

Контекст:
{rag_context}

Вопрос:
Как в документе определяется хроническая бедность и какие
характеристики хронической бедности выделяют авторы?

Ответ дай кратко и по существу.
"""

In [143]:
test_prompts = {
    "logic": logic_prompt,
    "code": code_prompt,
    "analytic": analytic_prompt,
    "rag": rag_prompt,
}

MODELS = [
    "gigachat",
    "openrouter_small",
    "openrouter_large",
]

Функция для запуска всех тестов:

In [144]:
benchmark_results = []
model_answers = {}

for task_name, prompt in test_prompts.items():

    model_answers[task_name] = {}

    for model_name in MODELS:

        print("\n" + "=" * 80)
        print(task_name, "|", model_name)

        try:
            result = call_model(model_name, prompt, max_tokens=1000)

            answer = result["text"]

            benchmark_results.append({
                "task": task_name,
                "model": model_name,
                "time_sec": result["time_sec"],
                "answer_chars": len(answer),
                "prompt_tokens": result["prompt_tokens"],
                "completion_tokens": result["completion_tokens"],
                "total_tokens": result["total_tokens"],
            })

            model_answers[task_name][model_name] = answer

            print(
                "Time:",
                round(result["time_sec"], 2),
                "sec"
            )
            print(
                "Tokens:",
                result["total_tokens"]
            )
            print(
                "Length:",
                len(answer),
                "chars"
            )

            print("\nAnswer:\n")
            print(answer)

        except Exception as e:
            print("ERROR:", repr(e))


logic | gigachat
Time: 5.91 sec
Tokens: 190
Length: 379 chars

Answer:

### Ход рассуждений:

Из условий известно следующее:

1. **Учитель** — не Маша и не Ира → значит учителем является Олег.
2. **Маша** не врач и не учитель → следовательно, профессия Маши — программист.
3. Осталась одна профессия (**врач**) и одно имя (**Ира**) → получается, Ира работает врачом.

---

### Ответ:

- **Маша** — программист  
- **Олег** — учитель  
- **Ира** — врач

logic | openrouter_small
Time: 4.3 sec
Tokens: 707
Length: 1046 chars

Answer:

**Решение**

| Человек | Профессия |
|--------|------------|
| Маша   | программист |
| Олег   | учитель   |
| Ира    | врач       |

**Как мы пришли к этому выводу**

1. **Маша ≠ врач** → Маша может быть либо *программистом*, либо *учителем*.
2. **Олег ≠ программист** → Олег может быть либо *учителем*, либо *врачом*.
3. **Ира ≠ учитель** → Ира может быть либо *программистом*, либо *врачом*.
4. **Учитель ≠ Маша** → Учителем может быть либо Олег, либо Ира.

Тепер

In [145]:
benchmark_df = pd.DataFrame(benchmark_results)
benchmark_df

,task,model,time_sec,answer_chars,prompt_tokens,completion_tokens,total_tokens
0,logic,gigachat,5.910866,379,79,111,190
1,logic,openrouter_small,4.302538,1046,75,632,707
2,logic,openrouter_large,11.327665,545,95,399,494
3,code,gigachat,8.606857,1332,70,335,405
4,code,openrouter_small,5.248955,1324,67,637,704
5,code,openrouter_large,3.215252,1726,88,638,726
6,analytic,gigachat,11.922416,2131,73,447,520
7,analytic,openrouter_small,10.393033,3195,66,998,1064
8,analytic,openrouter_large,9.300930,2671,87,839,926
9,rag,gigachat,4.393836,392,1494,89,1583


### Предварительная оценка качества

Все три модели корректно справились с четырьмя основными задачами.

- **Логическая задача:** все модели дали правильное решение. GigaChat ответил наиболее кратко, OpenRouter-модели — более подробно и структурированно.
- **Генерация кода:** все три модели предложили корректную реализацию функции и правильно обработали дубликаты и случай с менее чем двумя уникальными значениями.
- **Аналитический ответ:** все модели правильно объяснили назначение train, validation и test. OpenRouter-модели дали более подробные ответы, GigaChat — более компактный.
- **RAG-вопрос:** все модели корректно использовали предоставленный контекст и выделили основные характеристики хронической бедности. Явных галлюцинаций не обнаружено.

По скорости результаты различаются от задачи к задаче. GigaChat не всегда был самым быстрым, а крупная OpenRouter-модель в отдельных задачах работала быстрее маленькой. Поэтому по четырём запросам делать общий вывод о скорости пока рано.

### Тест длинного контекста

Все модели получают одинаковый prompt объёмом более 1000 входных токенов.  
Проверяется способность обработать длинный контекст, а также фиксируются время ответа и использование токенов.

In [146]:
long_context = rag_text[:12000]

long_prompt = f"""
Ниже приведён фрагмент научного документа.

Прочитай его и кратко ответь на три вопроса:

1. Как авторы определяют хроническую бедность?
2. Какие основные характеристики хронической бедности они выделяют?
3. Какие подходы к анализу бедности упоминаются в документе?

Используй только информацию из приведённого текста.
Если какой-либо информации недостаточно, прямо скажи об этом.

Текст:
{long_context}
"""

Запуск на всех моделях:

In [147]:
long_prompt_results = []

for model_name in MODELS:
    print("\n" + "=" * 80)
    print(model_name)

    try:
        result = call_model(model_name, long_prompt, max_tokens=1000)

        long_prompt_results.append({
            "model": model_name,
            "time_sec": result["time_sec"],
            "answer_chars": len(result["text"]),
            "prompt_tokens": result["prompt_tokens"],
            "completion_tokens": result["completion_tokens"],
            "total_tokens": result["total_tokens"],
        })

        print("Time:", round(result["time_sec"], 2), "sec")
        print("Prompt tokens:", result["prompt_tokens"])
        print("Completion tokens:", result["completion_tokens"])
        print("Total tokens:", result["total_tokens"])
        print("\nAnswer:\n")
        print(result["text"])

    except Exception as e:
        print("ERROR:", repr(e))


gigachat
Time: 26.07 sec
Prompt tokens: 2854
Completion tokens: 621
Total tokens: 3475

Answer:

### Ответы на вопросы:

**1. Определение хронической бедности авторами:**  
Авторы определяют хроническую бедность как состояние длительного пребывания человека или группы лиц в условиях бедности, часто начиная с рождения и продолжаясь всю жизнь. Хроническая бедность характеризуется длительностью, межпоколенной передачей, многоаспектностью проявлений и устойчивостью к существующим мерам борьбы с бедностью.

---

**2. Основные характеристики хронической бедности:**  
Авторы выделяют несколько ключевых характеристик хронической бедности:
- **Длительность**: длительный срок нахождения в состоянии бедности является необходимым и достаточным условием отнесения бедности к хронической.
- **Многоаспектность (multi-dimensionality)**: хронически бедные испытывают нехватку ресурсов не только в денежном выражении, но и в социальных, культурных, политических, человеческих и природных ресурсах.
- **Инте

In [148]:
long_prompt_df = pd.DataFrame(long_prompt_results)
long_prompt_df

,model,time_sec,answer_chars,prompt_tokens,completion_tokens,total_tokens
0,gigachat,26.074792,3055,2854,621,3475
1,openrouter_small,13.419111,2064,2623,998,3621
2,openrouter_large,5.743514,2064,2839,1000,3839


### Вывод по длинному prompt

Все три модели успешно обработали входной контекст объёмом более 2500 токенов.

В данном тесте самой быстрой оказалась крупная модель OpenRouter, затем малая модель OpenRouter и GigaChat.

При этом обе модели OpenRouter достигли установленного лимита `max_tokens=1000`, поэтому их ответы были частично обрезаны. GigaChat завершил ответ самостоятельно и не исчерпал лимит генерации.

Таким образом, ограничений по размеру контекста в рамках данного эксперимента не обнаружено, но при длинных ответах необходимо учитывать лимит `max_tokens`.

### Нагрузочный тест

Для каждой модели выполняется 10 последовательных коротких запросов.  
Фиксируются успешность запросов, среднее время ответа и возникающие ошибки API.

In [149]:
load_test_prompt = "Одним предложением объясни, что такое векторная база данных."

In [150]:
def run_load_test(model_name, n_requests=10):
    times = []
    errors = []

    for i in range(n_requests):
        try:
            result = call_model(model_name, load_test_prompt, max_tokens=300)

            times.append(result["time_sec"])

            print(
                f"{i + 1}/{n_requests}:",
                f"{result['time_sec']:.2f} sec"
            )

        except Exception as e:
            errors.append(repr(e))

            print(
                f"{i + 1}/{n_requests}: ERROR",
                repr(e)
            )

    return {
        "model": model_name,
        "requests": n_requests,
        "successful": len(times),
        "errors": len(errors),
        "avg_time_sec": np.mean(times) if times else None,
        "min_time_sec": np.min(times) if times else None,
        "max_time_sec": np.max(times) if times else None,
        "error_messages": errors,
    }

In [151]:
load_test_results = []

for model_name in MODELS:
    print("\n" + "=" * 80)
    print(model_name)

    result = run_load_test(model_name, n_requests=10)

    load_test_results.append(result)


gigachat
1/10: 2.80 sec
2/10: 3.62 sec
3/10: 1.24 sec
4/10: 1.41 sec
5/10: 0.95 sec
6/10: 1.08 sec
7/10: 1.29 sec
8/10: 5.11 sec
9/10: 2.39 sec
10/10: 1.16 sec

openrouter_small
1/10: 3.49 sec
2/10: 1.81 sec
3/10: 1.01 sec
4/10: 2.33 sec
5/10: 2.18 sec
6/10: 0.80 sec
7/10: 0.80 sec
8/10: 0.99 sec
9/10: 1.13 sec
10/10: 1.69 sec

openrouter_large
1/10: 1.69 sec
2/10: 1.62 sec
3/10: 23.97 sec
4/10: 3.48 sec
5/10: 3.22 sec
6/10: 2.48 sec
7/10: 24.68 sec
8/10: 1.12 sec
9/10: 2.41 sec
10/10: 1.16 sec


In [152]:
load_test_df = pd.DataFrame([
    {
        k: v
        for k, v in result.items()
        if k != "error_messages"
    }
    for result in load_test_results
])

load_test_df

,model,requests,successful,errors,avg_time_sec,min_time_sec,max_time_sec
0,gigachat,10,10,0,2.105180,0.954338,5.110277
1,openrouter_small,10,10,0,1.621438,0.795209,3.491048
2,openrouter_large,10,10,0,6.582635,1.123444,24.679016


### Итоговое сравнение моделей

| Критерий | GigaChat | OpenRouter small | OpenRouter large |
|---|---:|---:|---:|
| Логика | корректно | корректно | корректно |
| Генерация кода | корректно | корректно | корректно |
| Аналитический ответ | корректно | корректно | корректно |
| RAG | корректно | корректно | корректно |
| Явные галлюцинации | не обнаружены | не обнаружены | не обнаружены |
| Средняя задержка в нагрузочном тесте | 2.11 с | 1.62 с | 6.58 с |
| Минимальная задержка | 0.95 с | 0.80 с | 1.12 с |
| Максимальная задержка | 5.11 с | 3.49 с | 24.68 с |
| Успешных запросов | 10/10 | 10/10 | 10/10 |
| Long prompt 1000+ токенов | успешно | успешно | успешно |
| Удобство интеграции | хорошее | высокое | высокое |
| Стоимость в эксперименте | бесплатно по доступной квоте | бесплатно | бесплатно |
| Стабильность API | высокая | средняя | средняя |

### Вывод

Все три модели успешно справились с логической задачей, генерацией кода, аналитическим вопросом и RAG-вопросом по документу. Существенных ошибок или явных галлюцинаций в основных тестах не обнаружено.

GigaChat показал стабильную работу и обычно давал более компактные ответы. Интеграция через отдельный SDK немного специфичнее, чем у OpenRouter, но API в эксперименте работал стабильно.

OpenRouter удобен тем, что через единый OpenAI-совместимый интерфейс позволяет быстро переключаться между большим числом моделей. Это особенно удобно для экспериментов и выбора модели под конкретную задачу. При этом бесплатные модели оказались менее предсказуемыми по доступности: в ходе настройки встречались невалидные или снятые free-endpoint'ы, а также временные ошибки `429`.

Малая модель OpenRouter в нагрузочном тесте показала наименьшую среднюю задержку. Крупная модель также показала хорошее качество, но её время ответа было заметно менее стабильным.

На длинном prompt все модели успешно обработали контекст более 2500 токенов. Ограничений по длине контекста в рамках проведённого эксперимента не обнаружено, хотя OpenRouter-модели достигли установленного лимита генерации `max_tokens=1000`.

Для задач, где важны стабильность API, русскоязычные ответы и предсказуемая интеграция, GigaChat выглядит удобным вариантом. OpenRouter выгоднее, когда требуется доступ к разным моделям через единый интерфейс и возможность быстро сравнивать их между собой.

Локальный запуск оправдан, если важны конфиденциальность, отсутствие внешних API-зависимостей и полный контроль над моделью. Для небольших экспериментов и нерегулярных запросов API обычно проще, так как не требует собственной инфраструктуры и вычислительных ресурсов.

В данном эксперименте GigaChat использовался в рамках доступной квоты, а модели OpenRouter — через бесплатные endpoints, поэтому фактических денежных затрат не было.